# Linear discriminant analysis with empirical, Ledoit–Wolf and OAS covariance

The [scikit-learn gallery example](https://scikit-learn.org/dev/auto_examples/classification/plot_lda.html)
uses two classes with one informative feature and Gaussian noise features.
Each draw has 20 training rows and 200 test rows. The empirical within-class
covariance has rank at most 18, so this example exercises the `lsqr` solver on
rank-deficient data as the feature count increases.

We seed the draws with 0 and use 10 draws per feature count. Both libraries
classify the same rows. The Moxie example uses typed integer labels and a small
adapter for `moxie.covariance.OAS`.

Adapted from `examples/classification/plot_lda.py` at scikit-learn
`4358542a1bd39af2e7e1cbace8b9fed067473c5b` by the scikit-learn developers.
See the [BSD-3-Clause notice](../../docs/discriminant_analysis/THIRD_PARTY_NOTICES).

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "notebooks/_support/moxie_nb.py").exists())
sys.path.insert(0, str(ROOT / "notebooks/_support"))
%load_ext moxie_nb
nb = moxie_nb.Session("discriminant_analysis/plot_lda")

## 1. scikit-learn (Python reference)

The family oracle loads the pinned scikit-learn LDA and covariance sources.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import make_blobs

sys.path.insert(0, str(ROOT / "tests/discriminant_analysis"))
from oracle import load

pinned = load()
LinearDiscriminantAnalysis = pinned["da"].LinearDiscriminantAnalysis
OAS = pinned["shrunk"].OAS

n_train = 20  # samples for training
n_test = 200  # samples for testing
n_averages = 10  # how often to repeat classification (gallery: 50)
n_features_max = 75  # maximum number of features
step = 4  # step size for the calculation


def generate_data(n_samples, n_features):
    """Generate random blob-ish data with noisy features."""
    X, y = make_blobs(n_samples=n_samples, n_features=1, centers=[[-2], [2]])

    # add non-discriminative features
    if n_features > 1:
        X = np.hstack([X, np.random.randn(n_samples, n_features - 1)])
    return X, y


np.random.seed(0)  # adaptation: the gallery is unseeded
draws = {}  # adaptation: record every draw for Moxie
acc_clf1, acc_clf2, acc_clf3 = [], [], []
n_features_range = range(1, n_features_max + 1, step)
for n_features in n_features_range:
    score_clf1, score_clf2, score_clf3 = 0, 0, 0
    for rep in range(n_averages):
        X, y = generate_data(n_train, n_features)

        clf1 = LinearDiscriminantAnalysis(solver="lsqr", shrinkage=None).fit(X, y)
        clf2 = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto").fit(X, y)
        oa = OAS(store_precision=False, assume_centered=False)
        clf3 = LinearDiscriminantAnalysis(solver="lsqr", covariance_estimator=oa).fit(X, y)
        if rep == 0:
            draws[f"coef_{n_features}"] = np.vstack([clf1.coef_, clf2.coef_, clf3.coef_])
        draws.setdefault(f"Xtr_{n_features}", []).append(X)
        draws.setdefault(f"ytr_{n_features}", []).append(y)

        X, y = generate_data(n_test, n_features)
        draws.setdefault(f"Xte_{n_features}", []).append(X)
        draws.setdefault(f"yte_{n_features}", []).append(y)
        score_clf1 += clf1.score(X, y)
        score_clf2 += clf2.score(X, y)
        score_clf3 += clf3.score(X, y)

    acc_clf1.append(score_clf1 / n_averages)
    acc_clf2.append(score_clf2 / n_averages)
    acc_clf3.append(score_clf3 / n_averages)

features_samples_ratio = np.array(n_features_range) / n_train


def plot(ax, title, a1, a2, a3):
    ax.plot(features_samples_ratio, a1, linewidth=2, label="LDA", color="gold", linestyle="solid")
    ax.plot(features_samples_ratio, a2, linewidth=2, label="LDA with Ledoit Wolf", color="navy", linestyle="dashed")
    ax.plot(features_samples_ratio, a3, linewidth=2, label="LDA with OAS", color="red", linestyle="dotted")
    ax.set_xlabel("n_features / n_samples")
    ax.set_ylabel("Classification accuracy")
    ax.legend(loc="lower left", fontsize=7)
    ax.set_ylim((0.65, 1.0))
    ax.set_title(title, fontsize=9)


fig, ax = plt.subplots(figsize=(5, 3.5), dpi=60)
plot(ax, "LDA vs. LDA with Ledoit Wolf vs. LDA with OAS (1 discriminative feature)", acc_clf1, acc_clf2, acc_clf3)
plt.show()
shared = {k: (np.vstack(v) if k.startswith("X") else np.concatenate(v).astype(np.int64)) for k, v in draws.items()
          if not k.startswith("coef")}
nb.share(n_features=np.array(list(n_features_range), dtype=np.int64), **shared)

## 2. Moxie (native Mojo)

The same loop over the recorded draws through `moxie.discriminant_analysis`, with `moxie.covariance.OAS` behind a small adapter.

In [ ]:
%%mojo lda_shrinkage
from numojo import NDArray, Shape, zeros
from moxie.covariance import OAS
from moxie.discriminant_analysis import LinearDiscriminantAnalysis, CovarianceEstimator as LDACovariance
from moxie_nb_io import load, load_ints, save, save_floats

comptime Dense = NDArray[DType.float64]


struct OASCovariance(LDACovariance):
    """Adapter: moxie.covariance.OAS as a discriminant-analysis covariance_estimator."""
    var model: OAS[]

    def __init__(out self, *, store_precision: Bool, assume_centered: Bool):
        self.model = OAS(store_precision=store_precision, assume_centered=assume_centered)

    def fit(mut self, X: Dense) raises:
        _ = self.model.fit(X)

    def covariance(self) raises -> Dense:
        return self.model.covariance_()


def block(X: Dense, start: Int, count: Int) raises -> Dense:
    var p = X.shape[1]
    var out = zeros[DType.float64](Shape(count, p))
    for i in range(count):
        for j in range(p):
            out.unsafe_set(i * p + j, X.unsafe_get((start + i) * p + j))
    return out^


def labels(y: List[Int], start: Int, count: Int) -> List[Int]:
    var out = List[Int](capacity=count)
    for i in range(count):
        out.append(y[start + i])
    return out^


def main() raises:
    var n_train = 20
    var n_test = 200
    var n_averages = 10
    var feature_counts = load_ints("n_features")
    var acc1 = List[Float64]()
    var acc2 = List[Float64]()
    var acc3 = List[Float64]()
    for f in feature_counts:
        var tag = String(f)
        var Xtr = load("Xtr_" + tag)
        var ytr = load_ints("ytr_" + tag)
        var Xte = load("Xte_" + tag)
        var yte = load_ints("yte_" + tag)
        var s1: Float64 = 0
        var s2: Float64 = 0
        var s3: Float64 = 0
        for rep in range(n_averages):
            var X = block(Xtr, rep * n_train, n_train)
            var y = labels(ytr, rep * n_train, n_train)
            var clf1 = LinearDiscriminantAnalysis(solver="lsqr", shrinkage=None)
            _ = clf1.fit(X, y)
            var clf2 = LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")
            _ = clf2.fit(X, y)
            var oa = OASCovariance(store_precision=False, assume_centered=False)
            var clf3 = LinearDiscriminantAnalysis[DType.float64, Int, OASCovariance](solver="lsqr", covariance_estimator=oa^)
            _ = clf3.fit(X, y)
            if rep == 0:
                var coef = zeros[DType.float64](Shape(3, f))
                for j in range(f):
                    coef.unsafe_set(j, clf1.coef_.unsafe_get(j))
                    coef.unsafe_set(f + j, clf2.coef_.unsafe_get(j))
                    coef.unsafe_set(2 * f + j, clf3.coef_.unsafe_get(j))
                save("coef_" + tag, coef)
            var T = block(Xte, rep * n_test, n_test)
            var t = labels(yte, rep * n_test, n_test)
            s1 += clf1.score(T, t)
            s2 += clf2.score(T, t)
            s3 += clf3.score(T, t)
        acc1.append(s1 / Float64(n_averages))
        acc2.append(s2 / Float64(n_averages))
        acc3.append(s3 / Float64(n_averages))
    save_floats("acc1", acc1)
    save_floats("acc2", acc2)
    save_floats("acc3", acc3)
    print("LDA", acc1[len(acc1) - 1], " Ledoit-Wolf", acc2[len(acc2) - 1], " OAS", acc3[len(acc3) - 1], "at", feature_counts[len(feature_counts) - 1], "features")

The following probe checks the trait requirement for passing `moxie.covariance.OAS` directly.

In [ ]:
nb.note("covariance_estimator=moxie.covariance.OAS()", "GAP",
        "The discriminant-analysis CovarianceEstimator trait requires fit(X) and covariance(). "
        "OAS exposes fluent fit(X) and covariance_(). The example uses OASCovariance to bridge them.")

In [ ]:
%%mojo oas_direct --expect-fail
from moxie.covariance import OAS
from moxie.discriminant_analysis import LinearDiscriminantAnalysis


def main() raises:
    # sklearn: LinearDiscriminantAnalysis(solver="lsqr", covariance_estimator=OAS(store_precision=False))
    var clf = LinearDiscriminantAnalysis[DType.float64, Int, OAS[]](
        solver="lsqr", covariance_estimator=OAS(store_precision=False))

## 3. Side by side and parity

The table shows LDA, Ledoit–Wolf and OAS accuracy at every third feature-to-sample ratio.

In [ ]:
mx = nb.results["lda_shrinkage"]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), dpi=60)
plot(axes[0], "scikit-learn", acc_clf1, acc_clf2, acc_clf3)
plot(axes[1], "Moxie", mx["acc1"], mx["acc2"], mx["acc3"])
plt.show()
moxie_nb.side_by_side(
    sklearn={f"{r:.2f}": f"{a:.3f} / {b:.3f} / {c:.3f}" for r, a, b, c in zip(features_samples_ratio[::3], acc_clf1[::3], acc_clf2[::3], acc_clf3[::3])},
    moxie={f"{r:.2f}": f"{a:.3f} / {b:.3f} / {c:.3f}" for r, a, b, c in zip(features_samples_ratio[::3], mx["acc1"][::3], mx["acc2"][::3], mx["acc3"][::3])})
nb.compare("mean accuracy, LDA (lsqr, shrinkage=None)", np.array(acc_clf1), mx["acc1"], rtol=0, atol=1e-12,
           reason="atol covers floating-point summation of 10 accuracy scores")
nb.compare("mean accuracy, LDA with Ledoit-Wolf (shrinkage='auto')", np.array(acc_clf2), mx["acc2"], rtol=0, atol=1e-12,
           reason="atol covers floating-point summation of 10 accuracy scores")
nb.compare("mean accuracy, LDA with OAS covariance_estimator", np.array(acc_clf3), mx["acc3"], rtol=0, atol=1e-12,
           reason="atol covers floating-point summation of 10 accuracy scores")
for f in [1, 9, 17]:
    nb.compare(f"coef_ at n_features={f} (None / auto / OAS), first draw", draws[f"coef_{f}"], mx[f"coef_{f}"], rtol=1e-8, atol=1e-9,
               reason="Float64 lstsq of a well-posed (n_features < n_train - 2) covariance system; native vs LAPACK gelsd")
nb.report()
nb.assert_parity()